In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/spaceship-titanic/sample_submission.csv
/kaggle/input/spaceship-titanic/train.csv
/kaggle/input/spaceship-titanic/test.csv


In [2]:
import pandas as pd
import numpy as nm



In [3]:
train_df = pd.read_csv('../input/spaceship-titanic/train.csv')
test_df = pd.read_csv('../input/spaceship-titanic/test.csv')

In [4]:
test_df.shape

(4277, 13)

In [5]:
pas=test_df.PassengerId

In [6]:
train_df.head()
y=train_df.Transported

In [7]:
datas=train_df.drop(['Transported'],axis='columns')
datas=pd.concat([datas,test_df],axis=0)

In [8]:
datas.tail()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
4272,9266_02,Earth,True,G/1496/S,TRAPPIST-1e,34.0,False,0.0,0.0,0.0,0.0,0.0,Jeron Peter
4273,9269_01,Earth,False,NaN,TRAPPIST-1e,42.0,False,0.0,847.0,17.0,10.0,144.0,Matty Scheron
4274,9271_01,Mars,True,D/296/P,55 Cancri e,NaN,False,0.0,0.0,0.0,0.0,0.0,Jayrin Pore
4275,9273_01,Europa,False,D/297/P,NaN,NaN,False,0.0,2680.0,0.0,0.0,523.0,Kitakan Conale
4276,9277_01,Earth,True,G/1498/S,PSO J318.5-22,43.0,False,0.0,0.0,0.0,0.0,0.0,Lilace Leonzaley


In [9]:
datas=datas.drop(['Name','Cabin'],axis='columns')

In [10]:
nan_cols = [i for i in datas.columns if datas[i].isnull().any()]
nan_cols

['HomePlanet',
 'CryoSleep',
 'Destination',
 'Age',
 'VIP',
 'RoomService',
 'FoodCourt',
 'ShoppingMall',
 'Spa',
 'VRDeck']

In [11]:
col=['RoomService',
 'FoodCourt',
 'ShoppingMall',
 'Spa',
 'VRDeck','Age']
datas[col]=datas[col].fillna(datas[col].median())

In [12]:
nan_cols = [i for i in datas.columns if datas[i].isnull().any()]
nan_cols

['HomePlanet', 'CryoSleep', 'Destination', 'VIP']

In [13]:
nol=datas.CryoSleep.unique()
nol.size

3

In [14]:
datas.shape

(12970, 11)

In [15]:
datas=datas.drop(['PassengerId'],axis='columns')

In [16]:
nan_cols = [i for i in datas.columns if datas[i].isnull().any()]
nan_cols

['HomePlanet', 'CryoSleep', 'Destination', 'VIP']

In [17]:
datas.shape

(12970, 10)

In [18]:
datas.isna().sum()

HomePlanet      288
CryoSleep       310
Destination     274
Age               0
VIP             296
RoomService       0
FoodCourt         0
ShoppingMall      0
Spa               0
VRDeck            0
dtype: int64

In [19]:
datas['VIP'].value_counts()

False    12401
True       273
Name: VIP, dtype: int64

In [20]:
datas=datas.fillna({
       'HomePlanet':'Earth',
        'CryoSleep':'False',
     'Destination':'TRAPPIST-1e',
             'VIP':'False'
})

In [21]:
datas.isna().sum()


HomePlanet      0
CryoSleep       0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
dtype: int64

In [22]:
dataset = pd.get_dummies(datas, drop_first=True)

In [23]:
dataset.head(7)

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,CryoSleep_False,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True,VIP_False
0,39.0,0.0,0.0,0.0,0.0,0.0,1,0,0,0,0,1,0,0
1,24.0,109.0,9.0,25.0,549.0,44.0,0,0,0,0,0,1,0,0
2,58.0,43.0,3576.0,0.0,6715.0,49.0,1,0,0,0,0,1,1,0
3,33.0,0.0,1283.0,371.0,3329.0,193.0,1,0,0,0,0,1,0,0
4,16.0,303.0,70.0,151.0,565.0,2.0,0,0,0,0,0,1,0,0
5,44.0,0.0,483.0,0.0,291.0,0.0,0,0,0,0,1,0,0,0
6,26.0,42.0,1539.0,3.0,0.0,0.0,0,0,0,0,0,1,0,0


In [24]:
X=dataset.iloc[:8693,:]
test=dataset.iloc[8693:,:]

In [25]:
test.shape

(4277, 14)

In [26]:
y.head()

0    False
1     True
2    False
3    False
4     True
Name: Transported, dtype: bool

In [27]:
from sklearn.model_selection import train_test_split

train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.2, random_state=42)

In [28]:
from sklearn.model_selection import StratifiedKFold
folds = StratifiedKFold(n_splits=5)

In [29]:
from sklearn.model_selection import cross_val_score


In [30]:
from sklearn.svm import SVC
model = SVC(C=2)
model.fit(train_X,train_y)

SVC(C=2)

In [31]:
cross_val_score(model,X,y)

array([0.77400805, 0.77573318, 0.78895917, 0.80034522, 0.80322209])

In [32]:
from sklearn.linear_model import LogisticRegression
model_1=LogisticRegression()
model_1.fit(train_X,train_y)

/opt/conda/lib/python3.7/site-packages/sklearn/linear_model/_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG,


LogisticRegression()

In [33]:
cross_val_score(model_1,X, y)

/opt/conda/lib/python3.7/site-packages/sklearn/linear_model/_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG,
/opt/conda/lib/python3.7/site-packages/sklearn/linear_model/_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression


array([0.7826337 , 0.78435883, 0.77803335, 0.78596087, 0.80264672])

In [34]:
from sklearn.naive_bayes import GaussianNB
mode = GaussianNB()

In [35]:
cross_val_score(mode,X, y)

array([0.6986774 , 0.67970098, 0.71075331, 0.72497123, 0.69677791])

In [36]:
from sklearn.ensemble import RandomForestClassifier
modl = RandomForestClassifier(n_estimators=40)

In [37]:

modl.fit(train_X,train_y)

RandomForestClassifier(n_estimators=40)

In [38]:
cross_val_score(modl,X, y)

array([0.78435883, 0.78033353, 0.78723404, 0.783084  , 0.79976985])

In [39]:
from sklearn.ensemble import GradientBoostingClassifier #For Regression
cl = GradientBoostingClassifier(n_estimators=100, learning_rate=0.5, max_depth=3)
cl.fit(train_X,train_y)
cl.score(test_X,test_y)

0.7837837837837838

In [40]:
from sklearn.ensemble import AdaBoostClassifier #For Classification
from sklearn.ensemble import AdaBoostRegressor #For Regression
from sklearn.tree import DecisionTreeClassifier
dtree = DecisionTreeClassifier()
clr = AdaBoostClassifier(n_estimators=100, base_estimator=dtree,learning_rate=2)
clr.fit(train_X,train_y)
clr.score(test_X,test_y)

0.726279470960322

In [41]:
predicted=cl.predict(test)

In [42]:
prediction = pd.DataFrame(predicted,columns=['Transported']).to_csv('submissio.csv',index=False)
result=pd.read_csv('submissio.csv')
result=pd.concat([pas,result],axis='columns')
#result=result.drop([1459],axis=0)
result.tail()

,PassengerId,Transported
4272,9266_02,True
4273,9269_01,False
4274,9271_01,True
4275,9273_01,True
4276,9277_01,True


In [43]:
prediction = pd.DataFrame(result, columns=['PassengerId','Transported']).to_csv('submission.csv',index=False)
results=pd.read_csv('submission.csv')
results.tail()

,PassengerId,Transported
4272,9266_02,True
4273,9269_01,False
4274,9271_01,True
4275,9273_01,True
4276,9277_01,True
